In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")

In [ ]:
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
test_ids = test["Id"]

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.drop(columns=['PoolQC' , 'Fence' , 'MiscFeature' , 'FireplaceQu' , 'MasVnrType' , 'Alley'] , axis=1 , inplace=True)

In [ ]:
test.drop(columns=['PoolQC' , 'Fence' , 'MiscFeature' , 'FireplaceQu' , 'MasVnrType' , 'Alley'] , axis=1 , inplace=True)

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

In [ ]:
def correlation_heatmap(df, num_cols):
    corr = df[num_cols].corr()
    plt.figure(figsize=(20,10))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Heatmap')
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
correlation_heatmap(df,numeric_columns)

In [ ]:
df.drop(columns=['Id','MiscVal','YrSold' ,'MoSold' , 'BsmtFinSF2' , 'OverallCond' , 'LowQualFinSF' , 'BsmtHalfBath' , '3SsnPorch'] , axis=1 , inplace=True)

In [ ]:
test.drop(columns=['Id','MiscVal','YrSold' ,'MoSold' , 'BsmtFinSF2' , 'OverallCond' , 'LowQualFinSF' , 'BsmtHalfBath' , '3SsnPorch'] , axis=1 , inplace=True)

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
correlation_heatmap(df,numeric_columns)

In [ ]:


def missing_and_unique_info(df):
    missing_count = df.isnull().sum()
    missing_percent = round((missing_count / len(df)) * 100, 2)

    cols_with_missing = missing_count[missing_count > 0].index

    if len(cols_with_missing) == 0:
        print("any columns with missing values")
        return pd.DataFrame()

    null_count = df[cols_with_missing].isnull().sum()

    unique_counts = pd.Series(index=cols_with_missing, dtype='object')
    for col in cols_with_missing:
        if df[col].dtype == 'object' or  isinstance(df[col].dtype, pd.CategoricalDtype):
            unique_counts[col] = df[col].nunique(dropna=True)
        else:
            unique_counts[col] = np.nan

    report = pd.DataFrame({
        'Missing %': missing_percent[cols_with_missing],
        'Total Count': len(df),
        'null Count': null_count,
        'Unique (for object/categorical)': unique_counts
    })

    return report



In [ ]:
missing_and_unique_info(df)

In [ ]:
x = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [ ]:
x_test = test

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
def plot_boxplots(df, num_cols):
    for col in num_cols:
        plt.figure(figsize=(6,2))
        sns.boxplot(x=df[col].dropna())
        plt.title(f'Boxplot: {col}')
        plt.show()

In [ ]:
numeric_columns = x_train.select_dtypes(include=np.number).columns
plot_boxplots(x_train,numeric_columns)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
imputer.fit(x_train[numeric_columns])

x_train[numeric_columns] = imputer.transform(x_train[numeric_columns])
x_val[numeric_columns] = imputer.transform(x_val[numeric_columns])
x_test[numeric_columns] = imputer.transform(x_test[numeric_columns])


In [ ]:
categorical_columns = x_train.select_dtypes(include=['object','category','bool']).columns

imputer = SimpleImputer(strategy='most_frequent')
imputer.fit(x_train[categorical_columns])

x_train[categorical_columns] = imputer.transform(x_train[categorical_columns])
x_val[categorical_columns] = imputer.transform(x_val[categorical_columns])
x_test[categorical_columns] = imputer.transform(x_test[categorical_columns])

In [ ]:
missing_and_unique_info(x_val)

In [ ]:
import pandas as pd

def get_iqr_bounds(df, num_cols, k=3.5):
    bounds = {}
    for c in num_cols:
        Q1 = df[c].quantile(0.25)
        Q3 = df[c].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - k * IQR
        upper = Q3 + k * IQR
        bounds[c] = (lower, upper)
    return bounds


def remove_outliers_with_bounds(X, bounds, y=None, verbose=True):
    X = X.copy()
    mask = pd.Series(True, index=X.index)

    for c, (lower, upper) in bounds.items():
        mask &= (X[c] >= lower) & (X[c] <= upper)

    before = len(X)
    X_filtered = X[mask]
    after = len(X_filtered)

    if verbose:
        print(f"Removed {before - after} outlier rows (remaining: {after})")

    if y is not None:
        y_filtered = y.loc[mask]
        return X_filtered, y_filtered

    return X_filtered


In [ ]:
bounds = get_iqr_bounds(df = x_train , num_cols = numeric_columns , k=3.5)

In [ ]:
x_train , y_train= remove_outliers_with_bounds(x_train ,bounds ,y_train )

In [ ]:
x_val , y_val = remove_outliers_with_bounds(x_val ,bounds , y_val)

In [ ]:
def plot_numeric_distributions(df, num_cols, bins=50):
    for col in num_cols:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[col].dropna(), kde=True, bins=bins)
        plt.title(f'Distribution: {col}')
        plt.show()

In [ ]:
numeric_columns = x_train.select_dtypes(include=np.number).columns

plot_numeric_distributions(x_train, numeric_columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train[numeric_columns] = scaler.fit_transform(x_train[numeric_columns])
x_val[numeric_columns] = scaler.transform(x_val[numeric_columns])
x_test[numeric_columns] = scaler.transform(x_test[numeric_columns])

In [ ]:
x_train[categorical_columns]

In [ ]:
x_train['Street'].value_counts()

In [ ]:
x_train.drop(columns=['Street'] , axis=1 , inplace=True)

In [ ]:
x_val.drop(columns=['Street'] , axis=1 , inplace=True)
x_test.drop(columns=['Street'] , axis=1 , inplace=True)

In [ ]:
x_train['Utilities'].value_counts()

In [ ]:
x_train.drop(columns=['Utilities'] , axis=1 , inplace=True)
x_val.drop(columns=['Utilities'] , axis=1 , inplace=True)
x_test.drop(columns=['Utilities'] , axis=1 , inplace=True)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = x_train.select_dtypes(exclude=['int64', 'float64']).columns

encoder = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')

encoder.fit(x_train[cat_cols])

x_train_encoded = encoder.transform(x_train[cat_cols])
x_val_encoded = encoder.transform(x_val[cat_cols])
x_test_encoded = encoder.transform(x_test[cat_cols])

encoded_col_names = encoder.get_feature_names_out(cat_cols)
x_train_encoded = pd.DataFrame(x_train_encoded, columns=encoded_col_names, index=x_train.index)
x_val_encoded = pd.DataFrame(x_val_encoded, columns=encoded_col_names, index=x_val.index)
x_test_encoded = pd.DataFrame(x_test_encoded, columns=encoded_col_names, index=x_test.index)

x_train = x_train.drop(columns=cat_cols)
x_val = x_val.drop(columns=cat_cols)
x_test = x_test.drop(columns=cat_cols)

x_train = x_train.join(x_train_encoded)
x_val = x_val.join(x_val_encoded)
x_test = x_test.join(x_test_encoded)


In [ ]:
from xgboost import XGBRegressor
model = XGBRegressor(random_state=42, n_estimators=500 , learning_rate = 0.05)
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_val)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

score = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(y_pred)))

print(score)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_val, y_pred)
print(f"R² Score: {r2:.4f}")

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": y_pred
})


In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
submission